# 06 — Segment & Revenue-at-Risk Analysis

Combines `mart_user_risk_scores` (model output) with user features to produce:
- segment-level revenue at risk,
- high-value × high-risk quadrant,
- retention ROI scenario.

In [ ]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
DUCKDB_PATH = Path('../data/processed/kkbox.duckdb')
con = duckdb.connect(str(DUCKDB_PATH))

In [ ]:
scores = con.execute('SELECT * FROM analytics.mart_user_risk_scores').df()
feats  = con.execute('SELECT * FROM analytics.mart_user_churn_features').df()

joined = scores.merge(feats[['msno', 'age_band', 'city', 'latest_is_auto_renew',
                              'latest_plan_days', 'avg_discount_rate']],
                       on='msno', how='left')
print('Shape:', joined.shape)

## Top-line revenue at risk

In [ ]:
headline = {
    'total_users': len(joined),
    'expected_monthly_revenue': float(joined['expected_revenue'].sum()),
    'revenue_at_risk_total':    float(joined['revenue_at_risk'].sum()),
    'risk_share':               float(joined['revenue_at_risk'].sum() / max(joined['expected_revenue'].sum(), 1)),
    'critical_band_users':      int((joined['risk_band'] == 'critical').sum()),
    'high_band_users':          int((joined['risk_band'] == 'high').sum()),
}
for k, v in headline.items():
    print(f'{k:30s} {v:,.2f}' if isinstance(v, float) else f'{k:30s} {v:,}')

## Risk band summary

In [ ]:
band_summary = joined.groupby('risk_band').agg(
    user_count=('msno', 'count'),
    avg_churn_probability=('churn_probability', 'mean'),
    actual_churn_rate=('is_churn', 'mean'),
    expected_revenue=('expected_revenue', 'sum'),
    revenue_at_risk=('revenue_at_risk', 'sum'),
).reindex(['critical', 'high', 'medium', 'low'])
band_summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
band_summary['revenue_at_risk'].plot(kind='bar', ax=ax, color=['#d62728', '#ff7f0e', '#ffd54f', '#2ca02c'])
ax.set_ylabel('Revenue at risk')
ax.set_title('Revenue at risk by risk band')
plt.tight_layout()
plt.show()

## Segment-level revenue exposure

In [ ]:
city_num = pd.to_numeric(joined['city'], errors='coerce').fillna(-1)
auto_num = pd.to_numeric(joined['latest_is_auto_renew'], errors='coerce').fillna(-1)
joined['city_tier'] = np.where(city_num.isin([1, 5, 6, 13, 22]), 'tier_1',
                       np.where(city_num.isin([4, 11, 14, 15, 18]), 'tier_2', 'tier_3'))
joined['auto_renew_status'] = np.where(auto_num == 1, 'auto_renew_on', 'auto_renew_off')

segment_summary = joined.groupby(['city_tier', 'age_band', 'auto_renew_status']).agg(
    user_count=('msno', 'count'),
    churn_rate=('is_churn', 'mean'),
    avg_churn_prob=('churn_probability', 'mean'),
    revenue_at_risk=('revenue_at_risk', 'sum'),
).reset_index()
segment_summary = segment_summary[segment_summary['user_count'] >= 30]
segment_summary.sort_values('revenue_at_risk', ascending=False).head(15)

## High-value × high-risk quadrant

In [ ]:
joined['value_tier']  = pd.qcut(joined['expected_revenue'].rank(method='first'), 4, labels=['v_low', 'v_med', 'v_high', 'v_top'])
joined['risk_tier']   = pd.qcut(joined['churn_probability'].rank(method='first'), 4, labels=['r_low', 'r_med', 'r_high', 'r_top'])

quadrant = joined.groupby(['value_tier', 'risk_tier'], observed=True).agg(
    user_count=('msno', 'count'),
    revenue_at_risk=('revenue_at_risk', 'sum'),
).reset_index()

pivot = quadrant.pivot(index='value_tier', columns='risk_tier', values='revenue_at_risk').fillna(0)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='Reds', ax=ax)
ax.set_title('Revenue at risk by value × risk quadrant')
plt.tight_layout()
plt.show()

## Retention ROI scenario

Assume an offer of 30% discount on next renewal. Cost = 30% × expected_revenue.
Assume the offer reduces churn probability by `delta_p` for the targeted band.

In [ ]:
DISCOUNT_COST = 0.30
DELTA_P = 0.06  # assumed churn reduction (from A/B test design)

scenarios = []
for band in ['critical', 'high']:
    sub = joined[joined['risk_band'] == band]
    n = len(sub)
    cost = (sub['expected_revenue'] * DISCOUNT_COST).sum()
    saved_users = (sub['churn_probability'] * DELTA_P / sub['churn_probability']).sum()  # ~= n * DELTA_P
    saved_revenue = (sub['expected_revenue'] * DELTA_P).sum()
    net = saved_revenue - cost
    scenarios.append({
        'band': band,
        'targeted_users': n,
        'discount_cost': cost,
        'estimated_revenue_saved': saved_revenue,
        'net_value': net,
        'roi': net / max(cost, 1),
    })
scenarios = pd.DataFrame(scenarios)
scenarios

## Persist segment summary back to DuckDB

In [ ]:
con.register('segment_df', segment_summary)
con.execute('CREATE OR REPLACE TABLE analytics.mart_segment_action_summary AS SELECT * FROM segment_df')
con.register('scenarios_df', scenarios)
con.execute('CREATE OR REPLACE TABLE analytics.mart_retention_scenarios AS SELECT * FROM scenarios_df')
print('Wrote analytics.mart_segment_action_summary:', len(segment_summary))
print('Wrote analytics.mart_retention_scenarios:', len(scenarios))
con.close()